# pinn-kernels on a GPU (Colab)

Runtime → Change runtime type → **T4 / L4 / A100 GPU**. Runs the GPU half of the project: Triton kernels, LLVM NVPTX kernels, GPU tests, benchmarks, and GPU training. Results land in `results/`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 1. LLVM 18 dev headers (Colab is Ubuntu 22.04, which ships LLVM 14)

In [ ]:
!wget -qO llvm.sh https://apt.llvm.org/llvm.sh && sudo bash llvm.sh 18 > /dev/null
!sudo apt-get install -y -qq llvm-18-dev libzstd-dev libedit-dev > /dev/null
!/usr/lib/llvm-18/bin/llvm-config --version

## 2. Get the code
Either clone the GitHub repo or upload `pinn-kernels.zip` to the Colab file browser first.

In [ ]:
REPO_URL = 'https://github.com/sree-chi/pinn-kernels'  # change if the repo lives elsewhere
import os
if os.path.exists('pinn-kernels.zip'):
    !unzip -qo pinn-kernels.zip
elif not os.path.exists('pinn-kernels'):
    !git clone -q {REPO_URL}
%cd pinn-kernels

In [ ]:
!make build 2>&1 | tail -2
!pip install -q -e . cupy-cuda12x

## 3. Tests (GPU tests run here; they are skipped on CPU)

In [ ]:
!python -m pytest -q

## 4. Kernel benchmark: forward/adjoint time and effective bandwidth vs a tensor copy
Includes a Triton block-size sweep and the LLVM NVPTX kernels.

In [ ]:
!python -m pinn_kernels.bench kernels --device cuda
!cat results/bench_kernels_cuda.md

## 5. Training step time per physics-loss path

In [ ]:
!python -m pinn_kernels.bench train --device cuda --epochs 300
!cat results/bench_train_cuda.md

## 6. Full runs: inverse problem (recover alpha) on the GPU kernels

In [ ]:
for be in ['triton', 'ptx']:
    !python -m pinn_kernels.train --problem inverse --physics fd --backend {be} --device cuda --epochs 10000 --log-every 2000 --out results/inverse_{be}_cuda.json --plot results/inverse_{be}_cuda.png

In [ ]:
!python -m pinn_kernels.train --problem inverse --physics autograd --device cuda --epochs 10000 --log-every 2000 --out results/inverse_autograd_cuda.json

In [ ]:
from IPython.display import Image, display
display(Image('results/inverse_triton_cuda.png'))

## 7. Look at what the compiler produced

In [ ]:
!python -m pinn_kernels.inspect_ir --pde heat --emit ptx --sm sm_75 | head -60

## 8. Save results
Download `results.zip` and send the numbers back; only measured numbers go on the resume.

In [ ]:
!zip -qr results.zip results && ls -la results